In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [2]:
!pip -q install configparser langchain openai
!pip -q install transformers huggingface_hub 
!pip -q install chromadb

In [3]:
from langchain.llms.base import LLM
from langchain.llms.utils import enforce_stop_tokens
from langchain.llms import OpenAI
from langchain.tools import Tool
from langchain.agents import initialize_agent

In [4]:
#Then convert the transformers pipeline into fact_llm 
import os
os.environ["OPENAI_API_KEY"] = ""
fact_llm = OpenAI(temperature=0)

In [5]:
from langchain import PromptTemplate, HuggingFaceHub, LLMChain

template = """Question: {question}

Answer: Let's think step by step."""

prompt_open = PromptTemplate(template=template, input_variables=["question"])

open_chain = LLMChain(prompt=prompt_open,llm = fact_llm)

In [6]:
question = "Given a source ontology and a target ontology. What is the matching class for source ontology class sensor?"

print(open_chain.run(question))

 First, we need to identify the source ontology class for sensor. This can be done by searching for the term "sensor" in the source ontology. Once we have identified the source ontology class for sensor, we can then search for the corresponding class in the target ontology. The matching class for the source ontology class sensor in the target ontology will depend on the context and the specific terms used in the target ontology.


In [7]:
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
recurSplitter = RecursiveCharacterTextSplitter(chunk_size=100,
                                               chunk_overlap=20,
                                               length_function=len)

In [8]:
with open('initial.txt') as init:
  txt_init = init.read()

initial_docs = recurSplitter.create_documents([txt_init])

In [9]:
with open('lexical.txt') as lin:
  txt_lin = lin.read()
lexical_docs = recurSplitter.create_documents([txt_lin])

In [10]:
with open('graphical.txt') as spc:
  txt_spc = spc.read()
graphical_docs = recurSplitter.create_documents([txt_spc])

In [11]:
from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [12]:
from langchain.vectorstores import FAISS
initial_chroma = FAISS.from_documents(initial_docs, embeddings)
lexical_chroma = FAISS.from_documents(lexical_docs, embeddings)
graphical_chroma = FAISS.from_documents(graphical_docs,embeddings)

In [13]:
# Starting the custom tool route

In [14]:
initial_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=initial_chroma.as_retriever())

In [15]:
lexical_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=lexical_chroma.as_retriever())

In [16]:
graphical_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=graphical_chroma.as_retriever())

In [17]:
tools = [
    Tool(
        name = "Initial",
        func=initial_retriever.run,
        description="useful for when you need to answer initial information."
    ),
    # Tool(
    #     name = "Lexical",
    #     func=lexical_retriever.run,
    #     description="useful for when you need to answer lexical information."
    # ),
    Tool(
        name = "Graphical",
        func=graphical_retriever.run,
        description="useful for when you need to answer graphical information."
    ),
]

In [18]:
example_agent = initialize_agent(tools, 
                         fact_llm, 
                         agent="zero-shot-react-description", 
                         verbose=True)

In [19]:
example_agent.run("What is the equivalent class for chairman in target ontology? Answer: First check the initial information. If it does not exist, retrive all the lexical and graphical information of chairman in the source ontology and retrieve all classes and their lexical and graphical information in the target ontology. Finally, find the most similar class to chairman in target ontology .")



> Entering new AgentExecutor chain...
 I need to compare the lexical and graphical information of chairman in the source ontology to the classes in the target ontology
Action: Graphical
Action Input: Retrieve all the lexical and graphical information of chairman in the source ontology and retrieve all classes and their lexical and graphical information in the target ontology
Observation:  I don't know.
Thought: I need to compare the lexical and graphical information of chairman in the source ontology to the classes in the target ontology
Action: Initial
Action Input: Retrieve all the lexical and graphical information of chairman in the source ontology
Observation:  I don't know.
Thought: I need to compare the lexical and graphical information of chairman in the source ontology to the classes in the target ontology
Action: Graphical
Action Input: Retrieve all classes and their lexical and graphical information in the target ontology
Observation:  I don't know.
Thought: I need to compa

'Chairman in the source ontology is not equivalent to any of the classes in the target ontology.'

In [20]:
example_agent.run("What is the equivalent class for subject area in target ontology? Answer: First check the initial information. If it does not exist, retrive the information of subject area in the source ontology and retrieve all classes and their information in the target ontology. Finally, find the most similar class to subject area in target ontology .")



> Entering new AgentExecutor chain...
 I need to find the equivalent class for subject area in target ontology
Action: Initial
Action Input: Information of subject area in the source ontology
Observation:  I don't know.
Thought: I need to find the classes and their information in the target ontology
Action: Graphical
Action Input: Classes and their information in the target ontology
Observation:  The target ontology will have classes for participants, topic, tutorial, and review, and their associated information.
Thought: I need to find the most similar class to subject area in target ontology
Action: Graphical
Action Input: Classes and their information in the target ontology
Observation:  The target ontology will have classes for participants, topic, tutorial, and review, and their associated information.
Thought: I now know the final answer
Final Answer: The equivalent class for subject area in target ontology is the class that is most similar to the subject area in the source ont

'The equivalent class for subject area in target ontology is the class that is most similar to the subject area in the source ontology.'

In [24]:
example_agent.run("Please compare the similarity of the subject area in the source ontology with the paper in the target ontology. Think step by step. ")



> Entering new AgentExecutor chain...
 I need to compare the subject areas of the two ontologies
Action: Graphical
Action Input: The source and target ontologies
Observation:  The source ontology was written by a range reviewer, read by a range reviewer, and had details entered by a domain conference. The target ontology is not mentioned in the context, so I don't know.
Thought: I need to compare the paper in the target ontology to the source ontology
Action: Initial
Action Input: The paper in the target ontology
Observation:  The paper in the target ontology is equivalent to the paper in the source ontology.
Thought: I now know the final answer
Final Answer: The subject area in the source ontology is similar to the paper in the target ontology.

> Finished chain.


'The subject area in the source ontology is similar to the paper in the target ontology.'

In [26]:
example_agent.run("You are acting as an Ontology Matcher. Please list all the entities that are similiar or equivalent in the source ontology and the target ontology. Think step by step. ")



> Entering new AgentExecutor chain...
 I need to compare the two ontologies to find the entities that are similar or equivalent.
Action: Graphical
Action Input: The source and target ontologies
Observation:  The source ontology was written by a range reviewer, read by a range reviewer, and had details entered by a domain conference. The target ontology is not mentioned in the context, so I don't know.
Thought: I need to compare the two ontologies to find the entities that are similar or equivalent.
Action: Initial
Action Input: The source and target ontologies
Observation:  The source and target ontologies are equivalent in that person, conference, paper, and reviewer are all mapped to the same concept in both ontologies.
Thought: I now know the final answer
Final Answer: Person, conference, paper, and reviewer are all mapped to the same concept in both ontologies.

> Finished chain.


'Person, conference, paper, and reviewer are all mapped to the same concept in both ontologies.'